# Tools and Agents with LangChain and LangGraph

#LangChain #LangGraph #Tools #Agents #GenAI #ReAct

This notebook covers:
- Writing custom tools with the `@tool` decorator
- Wiring tools into a LangGraph ReAct agent
- Running the agent end to end

In [ ]:
pip install openai langchain langchain-community langchain-openai langgraph duckduckgo-search python-dotenv requests

## Imports

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

## Tool 1: Web Search Tool

#Tools

- A tool is just a Python function wrapped with `@tool`
- The docstring is what the LLM reads to decide when to call the tool
- Keep the docstring short, clear and accurate

In [ ]:
search = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """
    Simple web search tool using DuckDuckGo.
    Input: query string
    Output: search results
    """
    return search.run(query)

## Tool 2: GST Calculator Tool

#Tools

Bug fix from the original code:
- Function parameter was misspelled as `amaount`, but the function body used `amount`
- This caused a `NameError` at call time
- Fixed by renaming the parameter to `amount`

In [ ]:
@tool
def gst_calculator(amount: float) -> str:
    """
    Calculates 18% GST on a given amount.
    Input: amount in rupees
    Output: GST breakdown with total
    """
    gst = (amount * 18) / 100
    total = amount + gst

    return f"Amount: {amount} | GST (18%): {gst} | Total: {total}"

## How an Agent Uses Tools

#Agents #ReAct

```
                +-------------------+
                |    User Query      |
                +-------------------+
                          |
                          v
                +---------------------------+
                | LLM checks: tool needed?  |
                +---------------------------+
                     |              |
                    Yes             No
                     |              |
                     v              v
           +-----------------+   +----------------+
           |    Run Tool      |   | Return Answer  |
           +-----------------+   +----------------+
                     |
                     v
           +--------------------------+
           | Add tool result to chat  |
           +--------------------------+
                     |
                     v
           +--------------------------+
           |    Call LLM again        |
           +--------------------------+
                     |
                     v
           (loop back to "tool needed?"
            until the LLM is done)
```

Key points:
- The agent loop keeps checking if another tool call is needed
- Tool results are appended back into the message history
- The loop ends once the LLM returns a final answer without a tool call

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import requests
from langchain_core.tools import tool

## Tool 3: GitHub Repo Info Tool

#Tools

Bug fixes from the original code:
- Spelling fixed: `descritpion` to `description`, `repositoy` to `repository`, `seperately` to `separately`
- Logic kept the same, only the docstring and error message text cleaned up

In [ ]:
@tool
def get_repo_info(owner: str, repo: str) -> dict:
    """
    Look up a public GitHub repository and return its star count and description.
    Provide the owner (username or organization) and the repository name separately.
    """
    url = f"https://api.github.com/repos/{owner}/{repo}"
    response = requests.get(url, timeout=10)

    if response.status_code != 200:
        return {"error": f"Could not fetch info, status code {response.status_code}"}

    data = response.json()
    return {
        "stars": data.get("stargazers_count"),
        "description": data.get("description")
    }

## Tool 4: Popularity Evaluator Tool

#Tools

Bug fixes from the original code:
- Decorator was written as `#tool` (a comment) instead of `@tool`
- Because of this the function was never registered as a real tool, and the agent could not call it
- Spelling fixed: `evalaute` to `evaluate`, `repositoy` to `repository`, `answr` to `answer`, `cound` to `count`

In [ ]:
@tool
def evaluate_popularity(stars: int) -> str:
    """
    Given a GitHub star count, return a short answer on how popular the repository is.
    """
    if stars is None:
        return "Could not determine the popularity, star count is missing"

    if stars >= 20000:
        return f"Very popular. {stars} stars is a large, well-known project"

    if stars >= 1000:
        return f"Fairly popular. {stars} stars shows solid community adoption"

    return f"Niche or new. {stars} stars is still a small community"

## Creating the Agent

#Agents #LangGraph

- `create_react_agent` builds a ready made ReAct agent from an LLM and a list of tools
- The LLM decides on its own which tool to call and when to stop

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
agent = create_react_agent(
    model=llm,               # decides which tool to call
    tools=[get_repo_info, evaluate_popularity]   # tools available to the agent
)

## Testing the Agent

#Agents

In [ ]:
result = agent.invoke(
    {"messages": [("user", "is the repo NationalSecurityAgency / ghidra popular on github?")]}
)

final_message = result["messages"][-1].content
final_message

In [ ]:
result = agent.invoke(
    {"messages": [("user", "is the repo aldinokemal / go-whatsapp-web-multidevice popular on github?")]}
)

final_message = result["messages"][-1].content
final_message

In [ ]:
# Full message trace: shows every AIMessage, ToolMessage and the reasoning steps in between
result["messages"]

---

## Interview Questions: Tools

#InterviewPrep #Tools

**Q1. What is a tool in LangChain, and why does it need a docstring?**
A tool is a regular Python function wrapped with the `@tool` decorator so an LLM can call it. The docstring is not optional documentation, it is the description the LLM reads to decide when and how to call the function, so it must be accurate and specific.

**Q2. What happens if you forget the `@tool` decorator on a function you want the agent to use?**
The function stays a plain Python function. It is never registered as a callable tool, so the agent cannot see it or invoke it, even if you add it to the tools list.

**Q3. Why should tool functions have type hints on their parameters and return value?**
Type hints let LangChain build the correct input schema for the tool. Without them, the framework cannot validate or correctly pass arguments from the LLM's tool call to your function.

**Q4. What is the difference between a tool's docstring and inline code comments?**
The docstring is sent to the LLM as part of the tool's definition and directly affects tool selection. Inline comments are only for human readers and are never seen by the LLM.

**Q5. Can a tool call an external API or service? Give an example from this notebook.**
Yes. `get_repo_info` calls the GitHub REST API with `requests.get` to fetch star count and description for a repository. Tools commonly wrap APIs, databases, calculators, or search engines.

## Interview Questions: Agents

#InterviewPrep #Agents

**Q1. What is an agent, and how is it different from a plain LLM call?**
An agent is an LLM combined with a set of tools and a control loop. Instead of producing one fixed response, it can decide to call a tool, read the result, and decide again, repeating until it has enough information for a final answer.

**Q2. What does `create_react_agent` do?**
It builds a ready made ReAct style agent from a given LLM and a list of tools, handling the reasoning and tool calling loop internally so you do not have to write it by hand.

**Q3. In the agent loop, what happens to a tool's output after it runs?**
The tool's output is added back into the conversation as a `ToolMessage`. The LLM is then called again with this new message in context, so it can use the result to continue reasoning or produce a final answer.

**Q4. How does the agent decide which tool to call?**
The LLM reads the user's message along with the descriptions and docstrings of all available tools, then chooses the tool whose purpose matches the current need, and generates the arguments for it.

**Q5. What would happen if you gave the agent two tools with very similar or vague docstrings?**
The LLM could easily pick the wrong tool or call the wrong one first, since it relies entirely on the docstrings to distinguish tools. Clear, distinct, specific docstrings are essential for reliable tool selection.

## Interview Questions: LangGraph

#InterviewPrep #LangGraph

**Q1. What is LangGraph, and how does it relate to LangChain?**
LangGraph is a library for building stateful, graph based LLM applications, including agents. It sits on top of LangChain's core abstractions like tools and messages, and provides the runtime that manages state and the control flow between steps.

**Q2. Why is `create_react_agent` described as "prebuilt" in LangGraph?**
LangGraph lets you build custom graphs node by node for full control, but it also ships common patterns like the ReAct agent as ready made, prebuilt graphs so you do not need to construct the loop yourself for standard use cases.

**Q3. What is stored in the `messages` key of the agent's result, and why does it matter?**
It stores the full conversation history, including the user message, every AI message, every tool call, and every tool result in order. This full trace is useful for debugging, since it shows exactly which tools were called and why.